# Импорты

In [1]:
import os
import json
import time
from datetime import datetime, timedelta
from pathlib import Path

import yaml
import cv2
import numpy as np
from ultralytics import YOLO
import albumentations as A
import torch
from PIL import Image, ImageDraw, ImageFont

/home/himoza/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Конфигурация

In [2]:
RESULTS_DIR = "./train_result_new_data"
os.makedirs(RESULTS_DIR, exist_ok=True)

DATA_YAML = "./new_block_detection-2/data.yaml"

DEVICE = 0 if torch.cuda.is_available() else 'cpu'

EPOCHS = 200
IMGSZ = 640
PATIENCE = 10          # Early stopping: остановка через 10 эпох без улучшения
BATCH_SIZE = 32

## Функция обучения

In [3]:
def train_yolo(
    model_name: str,
    project_name: str,
    results_dir: str,
    data_yaml: str,
    epochs: str,
    batch_size: int,
    imgsz: int,
    patience: int,
    custom_augmentations: list = None
  ):
    """
      Обучение YOLO модели с аугментацией с отслеживанием времени

      Args:
          model_name: путь к модели или имя ('yolov8n.pt', etc.)
          project_name: имя проекта
          results_dir: директория для сохранения результатов
          data_yaml: путь к data.yaml
          epochs: количество эпох
          batch_size: размер батча
          imgsz: размер изображения
          patience: early stopping patience
          custom_augmentations: Список аугментаций

      Returns:
          results: результаты обучения
          training_time: время обучения в секундах
    """
    # Загрузка модели
    model = YOLO(model_name,)
    full_project_path = os.path.abspath(os.path.join(results_dir, project_name))

    # Время начала обучения
    start_time = time.time()
    start_datetime = datetime.now()

    print("\n" + "="*60)
    print(f"НАЧАЛО ОБУЧЕНИЯ: {project_name}")
    print("="*60)
    print(f"Время начала: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Путь сохранения: {full_project_path}")
    print(f"Эпохи: {epochs} | Patience: {patience}")
    print("="*60 + "\n")

    # Обучение
    results = model.train(
        data=data_yaml,
        epochs=epochs,
        batch=batch_size,
        imgsz=imgsz,
        device=DEVICE,
        project=full_project_path,   # абсолютный путь
        name="train",
        exist_ok=True,                  # Перезаписывать, если существует
        patience=patience,              # Встроенный patience в ultralytics
        save=True,                      # Сохранять веса
        save_period=10,                 # Сохранять каждые 10 эпох
        val=True,                       # Валидация после каждой эпохи
        plots=True,                     # Сохранять графики
        verbose=True,
        augmentations=custom_augmentations
    )

    # Время окончания обучения
    end_time = time.time()
    end_datetime = datetime.now()
    training_time_seconds = end_time - start_time

    # Форматирование времени
    training_time_formatted = str(timedelta(seconds=int(training_time_seconds)))
    hours = int(training_time_seconds // 3600)
    minutes = int((training_time_seconds % 3600) // 60)
    seconds = int(training_time_seconds % 60)

    # Сохранение конфигурации обучения
    config_info = {
        "project": project_name,
        "data_yaml": data_yaml,
        "epochs": epochs,
        "patience": patience,
        "batch_size": batch_size,
        "imgsz": imgsz,
        "device": str(DEVICE),
        "training_time": {
            "seconds": training_time_seconds,
            "formatted": training_time_formatted,
            "hours": hours,
            "minutes": minutes,
            "seconds": seconds
        },
        "start_time": start_datetime.isoformat(),
        "end_time": end_datetime.isoformat(),
        "total_epochs_completed": getattr(results, 'epoch', epochs) if hasattr(results, 'epoch') else epochs
    }

    # Вывод информации о времени обучения
    print("\n" + "="*60)
    print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
    print("="*60)
    print(f"Время начала:   {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Время окончания: {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Общее время:     {training_time_formatted} ({hours}ч {minutes}м {seconds}с)")
    print(f"Результаты:      {full_project_path}")
    print("="*60 + "\n")
    return results

## Функция валидации

In [4]:
def validate_best_model(
    project_name: str,
    results_dir: str,
    data_yaml: str,
    batch_size: int,
    imgsz: int
  ):
    """Валидация лучшей модели и сохранение метрик"""
    full_project_path = os.path.abspath(os.path.join(results_dir, project_name))
    best_model_path = os.path.join(full_project_path, 'train', 'weights', 'best.pt')

    if not os.path.exists(best_model_path):
        print("Best model not found!")
        return None

    model = YOLO(best_model_path)

    # Валидация с подробными метриками
    metrics = model.val(
        data=data_yaml,
        batch=batch_size,
        imgsz=imgsz,
        device=DEVICE,
        plots=True,
        save_json=True,
        save_hybrid=True,
        project=full_project_path,
        name="val", 
    )

    # Сохранение метрик в текстовый файл
    metrics_path = os.path.join(full_project_path, "validation_metrics.txt")
    with open(metrics_path, 'w') as f:
        f.write("="*50 + "\n")
        f.write("РЕЗУЛЬТАТЫ ВАЛИДАЦИИ МОДЕЛИ\n")
        f.write("="*50 + "\n\n")

        # Основные метрики
        f.write("ОСНОВНЫЕ МЕТРИКИ:\n")
        f.write(f"  mAP50-95: {metrics.box.map:.4f}\n")
        f.write(f"  mAP50:     {metrics.box.map50:.4f}\n")
        f.write(f"  mAP75:     {metrics.box.map75:.4f}\n\n")

        # Дополнительные метрики (если доступны)
        if hasattr(metrics.box, 'mp'):
            f.write("ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ:\n")
            f.write(f"  Precision: {metrics.box.mp:.4f}\n")
            f.write(f"  Recall:    {metrics.box.mr:.4f}\n\n")

        # Метрики по классам
        f.write("МЕТРИКИ ПО КЛАССАМ:\n")
        if hasattr(metrics.box, 'ap') and hasattr(metrics.box, 'ap_class_index'):
            for cls_id, ap_value in zip(metrics.box.ap_class_index, metrics.box.ap):
                f.write(f"  Class {cls_id}: mAP = {ap_value:.4f}\n")

        # Сводка
        f.write("\n" + "="*50 + "\n")
        f.write(f"Всего классов: {len(metrics.box.ap_class_index)}\n")
        f.write(f"Лучший класс: Class {metrics.box.ap_class_index[metrics.box.ap.argmax()]} "
                f"(mAP={metrics.box.ap.max():.4f})\n")
        f.write(f"Худший класс: Class {metrics.box.ap_class_index[metrics.box.ap.argmin()]} "
                f"(mAP={metrics.box.ap.min():.4f})\n")

    print(f"Validation metrics saved to: {metrics_path}")
    print(f"\nИтоговые метрики:")
    print(f"   mAP50-95: {metrics.box.map:.4f}")
    print(f"   mAP50: {metrics.box.map50:.4f}")
    print(f"   mAP75: {metrics.box.map75:.4f}")

    return metrics


## Функция сопоставления предсказания YOLO с ground truth разметкой

In [5]:
def visualize_predictions_vs_gt(
    project_name,
    results_dir,
    data_yaml_path,
    num_samples=20,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
):
    """
    Создает изображения сравнения ground truth и предсказаний YOLO

    Args:
        project_name: имя проекта
        results_dir: путь к результату
        data_yaml_path: путь к data.yaml с информацией о датасете
        num_samples: количество изображений для визуализации
        conf_threshold: порог уверенности для предсказаний
        iou_threshold: порог IoU для NMS
        img_size: размер изображения для модели
        seed: random seed для воспроизводимости
    """
    full_project_path = os.path.join(results_dir, project_name)
    best_model_path = os.path.join(full_project_path, 'train', 'weights', 'best.pt')
    output_dir = os.path.join(full_project_path, 'val', 'res')
    # Создаем выходную директорию
    os.makedirs(output_dir, exist_ok=True)

    # Загружаем модель
    model = YOLO(best_model_path)

    # Загружаем конфигурацию датасета
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    class_names = data_config['names']
    num_classes = len(class_names)

    # Генерируем цвета для классов (фиксированные для воспроизводимости)
    np.random.seed(42)
    colors = np.random.randint(0, 255, size=(num_classes, 3), dtype=np.uint8)

    # Загружаем список изображений из валидационного сета
    val_images_dir = data_config.get('val', data_config.get('valid'))[3:]

    if val_images_dir is None:
        raise ValueError("Не найден путь к валидационным изображениям в data.yaml")

    # Получаем список изображений
    if os.path.isabs(val_images_dir):
        val_path = Path(val_images_dir)
    else:
        val_path = Path(data_yaml_path).parent / val_images_dir

    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        image_files.extend(list(val_path.glob(ext)))

    print(val_path)
    if len(image_files) == 0:
        raise ValueError(f"Не найдены изображения в {val_path}")

    # Случайно выбираем num_samples изображений
    np.random.seed(seed)
    selected_images = np.random.choice(image_files,
                                       size=min(num_samples, len(image_files)),
                                       replace=False)

    # Функция для отрисовки боксов (GT или предсказания)
    def draw_boxes(img, boxes, labels=None, confidences=None, is_pred=False):
        """
        Рисует bounding boxes на изображении
        """
        img_copy = img.copy()
        h, w = img_copy.shape[:2]

        for i, box in enumerate(boxes):
            # Конвертируем нормализованные координаты в пиксельные
            if max(box) <= 1.0:  # если координаты нормализованы
                x1, y1, x2, y2 = box
                x1, y1 = int(x1 * w), int(y1 * h)
                x2, y2 = int(x2 * w), int(y2 * h)
            else:
                x1, y1, x2, y2 = map(int, box)

            # Определяем цвет и параметры
            if is_pred:
                # Для предсказаний используем цвет класса и пунктир
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    color = tuple(map(int, colors[class_id % len(colors)]))
                else:
                    color = (0, 255, 0)  # зеленый по умолчанию

                thickness = 2
                line_type = cv2.LINE_AA

                # Рисуем прямоугольник
                cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness, line_type)

                # Добавляем подпись с классом и уверенностью
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    class_name = class_names[class_id] if class_id < len(class_names) else f"Class {class_id}"

                    conf_str = ""
                    if confidences is not None and i < len(confidences):
                        conf_str = f" {confidences[i]:.2f}"

                    label_text = f"{class_name}{conf_str}"

                    # Рисуем фон для текста
                    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(img_copy, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
                    cv2.putText(img_copy, label_text, (x1, y1 - 3),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
            else:
                # Для GT используем синий цвет и сплошную линию
                color = (255, 0, 0)  # синий для GT
                thickness = 2
                line_type = cv2.LINE_AA

                cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness, line_type)

                # Добавляем подпись класса для GT
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    class_name = class_names[class_id] if class_id < len(class_names) else f"Class {class_id}"

                    label_text = class_name
                    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(img_copy, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
                    cv2.putText(img_copy, label_text, (x1, y1 - 3),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

        return img_copy

    # Обрабатываем каждое изображение
    for idx, img_path in enumerate(selected_images):
        print(f"Обработка {idx+1}/{len(selected_images)}: {img_path.name}")

        # Загружаем изображение
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Не удалось загрузить изображение: {img_path}")
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # === Получаем Ground Truth ===
        # Ищем соответствующий label файл
        label_path = None
        possible_label_dirs = [
            val_path.parent / 'labels' / val_path.name,  # если labels/val
            Path(data_config.get('labels', '')) / val_path.name if data_config.get('labels') else None,
            val_path.parent / 'labels'  # если labels в той же папке
        ]

        for label_dir in possible_label_dirs:
            if label_dir is not None and label_dir.exists():
                potential_label = label_dir / f"{img_path.stem}.txt"
                if potential_label.exists():
                    label_path = potential_label
                    break

        # Если не нашли в стандартных местах, ищем рядом с изображением
        if label_path is None:
            potential_label = img_path.parent / f"{img_path.stem}.txt"
            if potential_label.exists():
                label_path = potential_label

        gt_boxes = []
        gt_labels = []

        if label_path and label_path.exists():
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0])
                        # YOLO формат: class_id cx cy w h (все нормализовано)
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = cx - bw / 2
                        y1 = cy - bh / 2
                        x2 = cx + bw / 2
                        y2 = cy + bh / 2

                        gt_boxes.append([x1, y1, x2, y2])
                        gt_labels.append(class_id)

        # === Получаем предсказания ===
        results = model.predict(
            str(img_path),
            conf=conf_threshold,
            iou=iou_threshold,
            imgsz=img_size,
            verbose=False
        )

        pred_boxes = []
        pred_labels = []
        pred_confs = []

        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            if len(boxes) > 0:
                # Получаем координаты в формате xyxy
                xyxy_boxes = boxes.xyxy.cpu().numpy()
                class_ids = boxes.cls.cpu().numpy()
                confidences = boxes.conf.cpu().numpy()

                # Нормализуем координаты
                h, w = img.shape[:2]
                normalized_boxes = []
                for box in xyxy_boxes:
                    x1, y1, x2, y2 = box
                    normalized_boxes.append([x1/w, y1/h, x2/w, y2/h])

                pred_boxes = normalized_boxes
                pred_labels = class_ids.tolist()
                pred_confs = confidences.tolist()

        # === Создаем изображения с аннотациями ===
        gt_img = draw_boxes(img_rgb, gt_boxes, gt_labels, is_pred=False)
        pred_img = draw_boxes(img_rgb, pred_boxes, pred_labels, pred_confs, is_pred=True)

        # === Объединяем изображения side-by-side ===
        # Добавляем заголовки
        gt_img_pil = Image.fromarray(gt_img)
        pred_img_pil = Image.fromarray(pred_img)

        # Создаем объединенное изображение
        total_width = gt_img_pil.width + pred_img_pil.width
        max_height = max(gt_img_pil.height, pred_img_pil.height) + 40  # +40 для заголовка

        combined_img = Image.new('RGB', (total_width, max_height), color=(255, 255, 255))

        # Вставляем изображения
        combined_img.paste(gt_img_pil, (0, 40))
        combined_img.paste(pred_img_pil, (gt_img_pil.width, 40))

        # Добавляем текст заголовков
        draw = ImageDraw.Draw(combined_img)

        try:
            # Пытаемся использовать системный шрифт
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
        except:
            # Если шрифт не найден, используем стандартный
            font = ImageFont.load_default()

        # Заголовки
        draw.text((10, 10), "Ground Truth", fill=(255, 0, 0), font=font)
        draw.text((gt_img_pil.width + 10, 10), "Predictions", fill=(0, 255, 0), font=font)

        # Дополнительная информация: название файла и статистика
        stats_text = f"File: {img_path.name} | GT objects: {len(gt_boxes)} | Pred objects: {len(pred_boxes)}"
        draw.text((10, max_height - 25), stats_text, fill=(0, 0, 0), font=ImageFont.load_default())

        # Сохраняем результат
        output_filename = f"comparison_{idx+1:03d}_{img_path.stem}.jpg"
        output_path = os.path.join(output_dir, output_filename)
        combined_img.save(output_path, quality=95)

        print(f"  Сохранено: {output_filename}")
        print(f"  GT: {len(gt_boxes)} объектов, Predictions: {len(pred_boxes)} объектов")
        print()

    print(f"\n✅ Готово! Все изображения сохранены в: {output_dir}")
    print(f"Всего обработано изображений: {len(selected_images)}")

    return output_dir

## Функции оценки метрики Intersection / GT

In [6]:
def box_area(box):
    x1, y1, x2, y2 = box

    return max(0, x2 - x1) * max(0, y2 - y1)


def intersection_area(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    return (
        max(0, x2 - x1) *
        max(0, y2 - y1)
    )


def intersection_over_gt(pred_box, gt_box):

    intersection = intersection_area(
        pred_box,
        gt_box
    )

    gt_area = box_area(gt_box)

    if gt_area == 0:
        return 0.0

    return intersection / gt_area


def iou(box1, box2):

    intersection = intersection_area(box1, box2)

    area1 = box_area(box1)
    area2 = box_area(box2)

    union = area1 + area2 - intersection

    if union == 0:
        return 0.0

    return intersection / union


def yolo_to_xyxy(label, img_width, img_height):

    cls, xc, yc, w, h = label

    xc *= img_width
    yc *= img_height
    w *= img_width
    h *= img_height

    x1 = xc - w / 2
    y1 = yc - h / 2
    x2 = xc + w / 2
    y2 = yc + h / 2

    return [x1, y1, x2, y2], int(cls)

In [7]:
def evaluate_image(
    model,
    image_path,
    label_path,
    matching_iou_threshold=0.1
):

    # Prediction

    results = model(
        str(image_path),
        verbose=False
    )

    result = results[0]

    img_height, img_width = result.orig_shape

    pred_boxes = result.boxes.xyxy.cpu().numpy()
    pred_classes = result.boxes.cls.cpu().numpy().astype(int)

    # Ground Truth

    gt_boxes = []
    gt_classes = []

    if label_path.exists():

        with open(label_path, "r") as f:

            for line in f:

                values = list(
                    map(float, line.strip().split())
                )

                box, cls = yolo_to_xyxy(
                    values,
                    img_width,
                    img_height
                )

                gt_boxes.append(box)
                gt_classes.append(cls)

    # Matching

    matches = []

    used_predictions = set()

    for gt_idx, (gt_box, gt_class) in enumerate(
        zip(gt_boxes, gt_classes)
    ):

        best_iou = 0.0
        best_pred_idx = None

        for pred_idx, (pred_box, pred_class) in enumerate(
            zip(pred_boxes, pred_classes)
        ):

            if pred_idx in used_predictions:
                continue

            # Класс должен совпадать
            if pred_class != gt_class:
                continue

            current_iou = iou(
                pred_box,
                gt_box
            )

            if current_iou > best_iou:

                best_iou = current_iou
                best_pred_idx = pred_idx

        if (
            best_pred_idx is not None
            and best_iou >= matching_iou_threshold
        ):

            used_predictions.add(best_pred_idx)

            pred_box = pred_boxes[best_pred_idx]

            custom_score = intersection_over_gt(
                pred_box,
                gt_box
            )

            matches.append({
                "gt_idx": gt_idx,
                "pred_idx": best_pred_idx,
                "class": gt_class,
                "iou": best_iou,
                "intersection_gt": custom_score
            })

    scores = [
        match["intersection_gt"]
        for match in matches
    ]

    if len(scores) > 0:
        image_metric = np.mean(scores)
    else:
        image_metric = None

    return image_metric, matches

# Обучение модели с нуля
Обучаем модель на 100 изображениях и валидируем на 10

## YOLOv8n

### Конфигурация

In [7]:
model = 'yolov8n.pt'
project_name = "yolov8n_training_without_transfer_learning"

### Обучение

In [8]:
train_results = train_yolo(
    model,
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    EPOCHS,
    BATCH_SIZE,
    IMGSZ,
    PATIENCE
  )


НАЧАЛО ОБУЧЕНИЯ: yolov8n_training_without_transfer_learning
Время начала: 2026-10-07 09:32:36
Путь сохранения: /home/himoza/block_detect/train_result_new_data/yolov8n_training_without_transfer_learning
Эпохи: 200 | Patience: 10

New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./new_block_detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0

### Валидация

In [10]:
metrics = validate_best_model(
      project_name,
      RESULTS_DIR,
      DATA_YAML,
      BATCH_SIZE,
      IMGSZ,
    )

WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in the future.


Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1783.9±407.7 MB/s, size: 50.5 KB)
val: Scanning /home/himoza/block_detect/new_block_detection-2/valid/labels.cache... 11 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11/11 5.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 5.8it/s 0.2s
                   all         11        218      0.572      0.312      0.495      0.285
Speed: 0.2ms preprocess, 4.9ms inference, 0.0ms loss, 1.0ms postprocess per image
Saving /home/himoza/block_detect/train_result_new_data/yolov8n_training_without_transfer_learning/val/predictions.json...

Evaluating faster-coco-eval mAP...
Evaluate annotation type *bbox*
COCOeval_opt.evaluate() finished...
DONE (t=0.01s).
Accumulating evaluation results...
COCOeva

In [11]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=30,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
)

new_block_detection-2/valid/images
Обработка 1/11: vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  Сохранено: comparison_001_vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  GT: 5 объектов, Predictions: 0 объектов

Обработка 2/11: vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  Сохранено: comparison_002_vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  GT: 36 объектов, Predictions: 1 объектов

Обработка 3/11: vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  Сохранено: comparison_003_vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  GT: 18 объектов, Predictions: 0 объектов

Обработка 4/11: vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  Сохранено: comparison_004_vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  GT: 28 объектов, Predictions: 1 объектов

Обработка 5/11: vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg
  Сохранено: comparison_005_vid0499img1

'./train_result_new_data/yolov8n_training_without_transfer_learning/val/res'

Результаты плохии т.к мало данных

# Перенос обучения + fine tuning
Берем предобученную модель и дообучаем на новом наборе данных с маленьким шагом обучения

In [8]:
import os
import json
import time
from datetime import datetime, timedelta
from pathlib import Path
import shutil

import yaml
import cv2
import numpy as np
from ultralytics import YOLO
import albumentations as A
import torch
from PIL import Image, ImageDraw, ImageFont

## Функция валидации

In [9]:
def validate_model(model_path, data_yaml, split="val"):
    """
    Отдельная функция валидации модели
    """

    print(f"\nVALIDATING: {model_path}\n")

    model = YOLO(model_path)

    metrics = model.val(
        data=data_yaml,
        split=split,
        verbose=True
    )

    results = {
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr
    }

    print("\nValidation Metrics:")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")

    return results

## Функция обучения

In [10]:
def format_time(seconds):
    """
    Красивое форматирование времени
    """

    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)

    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

In [11]:
def train_two_stage_yolo(
    data_yaml,
    project_name,
    pretrained_weights="yolov8m.pt",
    output_dir="training_results",
    imgsz=640,
    batch=16,

    # Stage 1
    stage1_epochs=150,
    stage1_freeze=10,
    stage1_lr=1e-3,

    # Stage 2
    stage2_epochs=150,
    stage2_lr=1e-4,

    # Early stopping
    patience=20,

    device=0,

    custom_augmentations=None
):
    """
    Двухэтапное обучение YOLO:
    1. Frozen backbone training
    2. Full fine-tuning
    """
    # TOTAL TIMER

    total_start_time = time.time()

    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)

    # STAGE 1

    print("\n" + "=" * 60)
    print("STAGE 1: TRAIN HEAD WITH FROZEN BACKBONE")
    print("=" * 60)

    stage1_start_time = time.time()

    model = YOLO(pretrained_weights)

    stage1_project = os.path.abspath(os.path.join(output_dir, project_name, 'stage1'))

    model.train(
        data=data_yaml,

        epochs=stage1_epochs,

        imgsz=imgsz,
        batch=batch,

        freeze=stage1_freeze,

        optimizer="AdamW",
        lr0=stage1_lr,

        cos_lr=True,
        lrf=0.1,

        patience=patience,

        project=str(stage1_project),
        name="train",

        device=device,

        save=True,
        save_period=20,

        verbose=True,
        augmentations=custom_augmentations,
        mosaic=0.5
    )

    # STAGE 1 TIME

    stage1_training_time = time.time() - stage1_start_time

    print("\n" + "=" * 60)
    print(f"STAGE 1 TRAINING TIME: {format_time(stage1_training_time)}")
    print("=" * 60)

    # BEST MODEL STAGE 1

    best_stage1 = os.path.join(stage1_project, "train", "weights", "best.pt")

    print(f"\nBest Stage 1 Model: {best_stage1}")

    # VALIDATION AFTER STAGE 1

    print("\n" + "=" * 60)
    print("VALIDATION AFTER STAGE 1")
    print("=" * 60)

    stage1_validation_start = time.time()

    stage1_metrics = validate_model(
        model_path=str(best_stage1),
        data_yaml=data_yaml
    )

    stage1_validation_time = time.time() - stage1_validation_start

    print(
        f"\nStage 1 Validation Time: "
        f"{format_time(stage1_validation_time)}"
    )

    # STAGE 2

    print("\n" + "=" * 60)
    print("STAGE 2: FULL FINE-TUNING")
    print("=" * 60)

    stage2_start_time = time.time()

    model_stage2 = YOLO(str(best_stage1))

    stage2_project = os.path.abspath(os.path.join(output_dir, project_name, 'stage2'))

    model_stage2.train(
        data=data_yaml,

        epochs=stage2_epochs,

        imgsz=imgsz,
        batch=batch,

        freeze=None,

        optimizer="AdamW",
        lr0=stage2_lr,

        cos_lr=True,
        lrf=0.1,

        patience=patience,

        project=str(stage2_project),
        name="train",

        device=device,

        save=True,
        save_period=20,

        verbose=True,
        augmentations=custom_augmentations,
        mosaic=0.5
    )

    # STAGE 2 TIME

    stage2_training_time = time.time() - stage2_start_time

    print("\n" + "=" * 60)
    print(f"STAGE 2 TRAINING TIME: {format_time(stage2_training_time)}")
    print("=" * 60)

    # BEST MODEL STAGE 2
    best_stage2 = os.path.join(stage2_project, "train", "weights", "best.pt")

    print(f"\nBest Stage 2 Model: {best_stage2}")

    # FINAL VALIDATION

    print("\n" + "=" * 60)
    print("FINAL VALIDATION")
    print("=" * 60)

    final_validation_start = time.time()

    final_metrics = validate_model(
        model_path=str(best_stage2),
        data_yaml=data_yaml
    )

    final_validation_time = time.time() - final_validation_start

    print(
        f"\nFinal Validation Time: "
        f"{format_time(final_validation_time)}"
    )

    # COPY FINAL MODEL

    final_model_path = os.path.join(output_dir, project_name, "best_final_model.pt")

    shutil.copy(best_stage2, final_model_path)

    print(f"\nFinal model saved to: {final_model_path}")

    # TOTAL TIME

    total_training_time = time.time() - total_start_time

    print("\n" + "=" * 60)
    print("TOTAL TRAINING SUMMARY")
    print("=" * 60)

    print(f"Stage 1 Training Time : {format_time(stage1_training_time)}")
    print(f"Stage 1 Validation Time: {format_time(stage1_validation_time)}")

    print(f"Stage 2 Training Time : {format_time(stage2_training_time)}")
    print(f"Final Validation Time : {format_time(final_validation_time)}")

    print(f"\nTOTAL PIPELINE TIME: {format_time(total_training_time)}")

    print("=" * 60)

    # RETURN RESULTS

    results = {
        "stage1_best_model": str(best_stage1),
        "stage2_best_model": str(best_stage2),
        "final_model": str(final_model_path),

        "stage1_metrics": stage1_metrics,
        "final_metrics": final_metrics,

        "timing": {
            "stage1_training_time_sec": stage1_training_time,
            "stage1_validation_time_sec": stage1_validation_time,
            "stage2_training_time_sec": stage2_training_time,
            "final_validation_time_sec": final_validation_time,
            "total_pipeline_time_sec": total_training_time,

            "stage1_training_time_str": format_time(stage1_training_time),
            "stage1_validation_time_str": format_time(stage1_validation_time),
            "stage2_training_time_str": format_time(stage2_training_time),
            "final_validation_time_str": format_time(final_validation_time),
            "total_pipeline_time_str": format_time(total_training_time)
        }
    }

    return results

## Функция сопоставления предсказания YOLO с ground truth разметкой

In [12]:
def visualize_predictions_vs_gt(
    project_name,
    results_dir,
    data_yaml_path,
    num_samples=20,
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=640,
    seed=42
):
    """
    Создает изображения сравнения ground truth и предсказаний YOLO

    Args:
        project_name: имя проекта
        results_dir: путь к результату
        data_yaml_path: путь к data.yaml с информацией о датасете
        num_samples: количество изображений для визуализации
        conf_threshold: порог уверенности для предсказаний
        iou_threshold: порог IoU для NMS
        img_size: размер изображения для модели
        seed: random seed для воспроизводимости
    """
    full_project_path = os.path.join(results_dir, project_name)
    best_model_path = os.path.join(full_project_path, 'stage2', 'train', 'weights', 'best.pt')
    output_dir = os.path.join(full_project_path, 'val', 'res')
    # Создаем выходную директорию
    os.makedirs(output_dir, exist_ok=True)

    # Загружаем модель
    model = YOLO(best_model_path)

    # Загружаем конфигурацию датасета
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    class_names = data_config['names']
    num_classes = len(class_names)

    # Генерируем цвета для классов (фиксированные для воспроизводимости)
    np.random.seed(42)
    colors = np.random.randint(0, 255, size=(num_classes, 3), dtype=np.uint8)

    # Загружаем список изображений из валидационного сета
    val_images_dir = data_config.get('val', data_config.get('valid'))[3:]

    if val_images_dir is None:
        raise ValueError("Не найден путь к валидационным изображениям в data.yaml")

    # Получаем список изображений
    if os.path.isabs(val_images_dir):
        val_path = Path(val_images_dir)
    else:
        val_path = Path(data_yaml_path).parent / val_images_dir

    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        image_files.extend(list(val_path.glob(ext)))

    print(val_path)
    if len(image_files) == 0:
        raise ValueError(f"Не найдены изображения в {val_path}")

    # Случайно выбираем num_samples изображений
    np.random.seed(seed)
    selected_images = np.random.choice(image_files,
                                       size=min(num_samples, len(image_files)),
                                       replace=False)

    # Функция для отрисовки боксов (GT или предсказания)
    def draw_boxes(img, boxes, labels=None, confidences=None, is_pred=False):
        """
        Рисует bounding boxes на изображении
        """
        img_copy = img.copy()
        h, w = img_copy.shape[:2]

        for i, box in enumerate(boxes):
            # Конвертируем нормализованные координаты в пиксельные
            if max(box) <= 1.0:  # если координаты нормализованы
                x1, y1, x2, y2 = box
                x1, y1 = int(x1 * w), int(y1 * h)
                x2, y2 = int(x2 * w), int(y2 * h)
            else:
                x1, y1, x2, y2 = map(int, box)

            # Определяем цвет и параметры
            if is_pred:
                # Для предсказаний используем цвет класса и пунктир
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    color = tuple(map(int, colors[class_id % len(colors)]))
                else:
                    color = (0, 255, 0)  # зеленый по умолчанию

                thickness = 2
                line_type = cv2.LINE_AA

                # Рисуем прямоугольник
                cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness, line_type)

                # Добавляем подпись с классом и уверенностью
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    class_name = class_names[class_id] if class_id < len(class_names) else f"Class {class_id}"

                    conf_str = ""
                    if confidences is not None and i < len(confidences):
                        conf_str = f" {confidences[i]:.2f}"

                    label_text = f"{class_name}{conf_str}"

                    # Рисуем фон для текста
                    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(img_copy, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
                    cv2.putText(img_copy, label_text, (x1, y1 - 3),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)
            else:
                # Для GT используем синий цвет и сплошную линию
                color = (255, 0, 0)  # синий для GT
                thickness = 2
                line_type = cv2.LINE_AA

                cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness, line_type)

                # Добавляем подпись класса для GT
                if labels is not None and i < len(labels):
                    class_id = int(labels[i])
                    class_name = class_names[class_id] if class_id < len(class_names) else f"Class {class_id}"

                    label_text = class_name
                    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(img_copy, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
                    cv2.putText(img_copy, label_text, (x1, y1 - 3),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

        return img_copy

    # Обрабатываем каждое изображение
    for idx, img_path in enumerate(selected_images):
        print(f"Обработка {idx+1}/{len(selected_images)}: {img_path.name}")

        # Загружаем изображение
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Не удалось загрузить изображение: {img_path}")
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # === Получаем Ground Truth ===
        # Ищем соответствующий label файл
        label_path = None
        possible_label_dirs = [
            val_path.parent / 'labels' / val_path.name,  # если labels/val
            Path(data_config.get('labels', '')) / val_path.name if data_config.get('labels') else None,
            val_path.parent / 'labels'  # если labels в той же папке
        ]

        for label_dir in possible_label_dirs:
            if label_dir is not None and label_dir.exists():
                potential_label = label_dir / f"{img_path.stem}.txt"
                if potential_label.exists():
                    label_path = potential_label
                    break

        # Если не нашли в стандартных местах, ищем рядом с изображением
        if label_path is None:
            potential_label = img_path.parent / f"{img_path.stem}.txt"
            if potential_label.exists():
                label_path = potential_label

        gt_boxes = []
        gt_labels = []

        if label_path and label_path.exists():
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0])
                        # YOLO формат: class_id cx cy w h (все нормализовано)
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = cx - bw / 2
                        y1 = cy - bh / 2
                        x2 = cx + bw / 2
                        y2 = cy + bh / 2

                        gt_boxes.append([x1, y1, x2, y2])
                        gt_labels.append(class_id)

        # === Получаем предсказания ===
        results = model.predict(
            str(img_path),
            conf=conf_threshold,
            iou=iou_threshold,
            imgsz=img_size,
            verbose=False
        )

        pred_boxes = []
        pred_labels = []
        pred_confs = []

        if len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes

            if len(boxes) > 0:
                # Получаем координаты в формате xyxy
                xyxy_boxes = boxes.xyxy.cpu().numpy()
                class_ids = boxes.cls.cpu().numpy()
                confidences = boxes.conf.cpu().numpy()

                # Нормализуем координаты
                h, w = img.shape[:2]
                normalized_boxes = []
                for box in xyxy_boxes:
                    x1, y1, x2, y2 = box
                    normalized_boxes.append([x1/w, y1/h, x2/w, y2/h])

                pred_boxes = normalized_boxes
                pred_labels = class_ids.tolist()
                pred_confs = confidences.tolist()

        # === Создаем изображения с аннотациями ===
        gt_img = draw_boxes(img_rgb, gt_boxes, gt_labels, is_pred=False)
        pred_img = draw_boxes(img_rgb, pred_boxes, pred_labels, pred_confs, is_pred=True)

        # === Объединяем изображения side-by-side ===
        # Добавляем заголовки
        gt_img_pil = Image.fromarray(gt_img)
        pred_img_pil = Image.fromarray(pred_img)

        # Создаем объединенное изображение
        total_width = gt_img_pil.width + pred_img_pil.width
        max_height = max(gt_img_pil.height, pred_img_pil.height) + 40  # +40 для заголовка

        combined_img = Image.new('RGB', (total_width, max_height), color=(255, 255, 255))

        # Вставляем изображения
        combined_img.paste(gt_img_pil, (0, 40))
        combined_img.paste(pred_img_pil, (gt_img_pil.width, 40))

        # Добавляем текст заголовков
        draw = ImageDraw.Draw(combined_img)

        try:
            # Пытаемся использовать системный шрифт
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
        except:
            # Если шрифт не найден, используем стандартный
            font = ImageFont.load_default()

        # Заголовки
        draw.text((10, 10), "Ground Truth", fill=(255, 0, 0), font=font)
        draw.text((gt_img_pil.width + 10, 10), "Predictions", fill=(0, 255, 0), font=font)

        # Дополнительная информация: название файла и статистика
        stats_text = f"File: {img_path.name} | GT objects: {len(gt_boxes)} | Pred objects: {len(pred_boxes)}"
        draw.text((10, max_height - 25), stats_text, fill=(0, 0, 0), font=ImageFont.load_default())

        # Сохраняем результат
        output_filename = f"comparison_{idx+1:03d}_{img_path.stem}.jpg"
        output_path = os.path.join(output_dir, output_filename)
        combined_img.save(output_path, quality=95)

        print(f"  Сохранено: {output_filename}")
        print(f"  GT: {len(gt_boxes)} объектов, Predictions: {len(pred_boxes)} объектов")
        print()

    print(f"\n✅ Готово! Все изображения сохранены в: {output_dir}")
    print(f"Всего обработано изображений: {len(selected_images)}")

    return output_dir

## Предобученная модель yolov8n

### Обучение

In [64]:
pretrained_weights = './train_result/yolov8n_training/train/weights/best.pt'
project_name = "yolov8n_training_transfer_learning"

In [65]:
results = train_two_stage_yolo(
    data_yaml=DATA_YAML,
    project_name=project_name,
    pretrained_weights=pretrained_weights,

    output_dir=RESULTS_DIR,

    imgsz=640,
    batch=BATCH_SIZE,

    stage1_epochs=200,
    stage1_freeze=10,
    stage1_lr=1e-4,

    stage2_epochs=200,
    stage2_lr=1e-5,

    patience=PATIENCE,

    device=DEVICE
)


STAGE 1: TRAIN HEAD WITH FROZEN BACKBONE
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./new_block_detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=./train_result/yolov8n_training/train/weig

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1896.4±578.8 MB/s, size: 49.5 KB)
train: Scanning /home/himoza/block_detect/new_block_detection-2/train/labels.cache... 98 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 98/98 27.4Mit/s 0.0s
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2211.3±671.1 MB/s, size: 48.2 KB)
val: Scanning /home/himoza/block_detect/new_block_detection-2/valid/labels.cache... 11 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11/11 4.6Mit/s 0.0s
optimizer: AdamW(lr=0.0001, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /home/himoza/block_detect/train_result_new_data/yolov8n_training_transfer_learning/stage1/train/labels.jpg... 
Using 98 tra

### Визуализация на новом наборе данных

In [66]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

new_block_detection-2/valid/images
Обработка 1/11: vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  Сохранено: comparison_001_vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  GT: 5 объектов, Predictions: 16 объектов

Обработка 2/11: vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  Сохранено: comparison_002_vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  GT: 36 объектов, Predictions: 62 объектов

Обработка 3/11: vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  Сохранено: comparison_003_vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  GT: 18 объектов, Predictions: 32 объектов

Обработка 4/11: vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  Сохранено: comparison_004_vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  GT: 28 объектов, Predictions: 54 объектов

Обработка 5/11: vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg
  Сохранено: comparison_005_vid0499

'./train_result_new_data/yolov8n_training_transfer_learning/val/res'

### Визуализация на старом наборе данных

In [67]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    './block-detection-1/data.yaml',
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/123: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/123: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 37 объектов

Обработка 3/123: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 40 объектов

Обработка 4/123: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/123: _barhat-2-500x500-500x500_jp

'./train_result_new_data/yolov8n_training_transfer_learning/val/res'

### Валидация на старом наборе данных

In [68]:
validate_model('./train_result_new_data/yolov8n_training_transfer_learning/stage2/train/weights/best.pt',
               './block-detection-1/data.yaml')


VALIDATING: ./train_result_new_data/yolov8n_training_transfer_learning/stage2/train/weights/best.pt

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2140.1±468.1 MB/s, size: 86.7 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 46.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.9it/s 4.1s0.2ss
                   all        123       8279      0.818      0.841      0.843      0.392
Speed: 2.1ms preprocess, 13.6ms inference, 0.0ms loss, 8.8ms postprocess per image
Results saved to /home/himoza/block_detect/runs/detect/val-4

Validation Metrics:
mAP50: 0.8426
mAP50-95: 0.3919
precision: 0.8176
recall: 0.8414


{'mAP50': np.float64(0.8426425074674702),
 'mAP50-95': np.float64(0.391926343214183),
 'precision': np.float64(0.8175986112606362),
 'recall': np.float64(0.8413663488016105)}

### Валидация на нашей метрике Intersection / Ground Truth на новом наборе данных

In [10]:
model = YOLO("./train_result_new_data/yolov8n_training_transfer_learning/best_final_model.pt")

images_dir = Path("./new_block_detection-2/valid/images")
labels_dir = Path("./new_block_detection-2/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg: 0.9714 объектов: 36
vid0493img330_3_2_jpg.rf.9b32fb16517572340e510438f4a2b700.jpg: 0.9443 объектов: 27
vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg: 0.8902 объектов: 10
vid0499img1380_4_1_jpg.rf.7ac56021fb5d32f299ee02e2f3f942e4.jpg: 0.8228 объектов: 8
vid0489img2280_1_2_jpg.rf.3f1bd4eae4d057439a2f91baf1de3333.jpg: 0.9489 объектов: 36
vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg: 0.7322 объектов: 5
vid0498img1110_4_2_jpg.rf.655393520c50bf01898d6ed9653a076c.jpg: 0.9848 объектов: 15
vid0498img30_4_2_jpg.rf.285af3e44d3400585ee7668aace70aad.jpg: 0.9822 объектов: 3
vid0489img2250_2_1_jpg.rf.06985a295a14e641f58c90652bc17b55.jpg: 0.9815 объектов: 32
vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg: 0.9711 объектов: 18
vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg: 0.9809 объектов: 28

Intersection / GT: 0.9534
Количество объектов: 218


### Валидация на нашей метрике Intersection / Ground Truth на старом наборе данных

In [11]:
model = YOLO("./train_result_new_data/yolov8n_training_transfer_learning/best_final_model.pt")

images_dir = Path("./block-detection-1/valid/images")
labels_dir = Path("./block-detection-1/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

many-stack-brick-background-construction-site-69549698_jpg.rf.2c1d4a0251603f35b6e7241f23e36a28.jpg: 0.9895 объектов: 136
vandersanden-kirpich-ruchnoy-formovki-oud-kapelle-m50-190-90-50-1_0001_w500_h500_watermark-2-_jpg.rf.8c61c216f6a48614049e8bea63053d13.jpg: 0.9993 объектов: 48
stack-bricks-red-construction-36132343_jpg.rf.a3ab585f8df68d8aa172d0b98a9d0ab1.jpg: 0.9581 объектов: 48
brick2-17-_jpg.rf.f64be790718963275d995fa4b939e546.jpg: 0.9833 объектов: 87
_barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg: 0.9985 объектов: 28
383_jpg.rf.bbc945440a6bb651e33008744c19bcd2.jpg: 0.8775 объектов: 27
4-1-_jpg.rf.38692d9e8160c842930d08b826f87182.jpg: 0.9988 объектов: 104
full_4404479-0-1-_jpg.rf.3fffd358b0471fd970c05ddeac0945e5.jpg: 0.9900 объектов: 42
stack-bricks-construction-territory-near-future-house-72727138_jpg.rf.4dcad82f85499f905a9cf3aa286f0c44.jpg: 0.9776 объектов: 57
unnamed_jpg.rf.2843209a631061453c3a09b086990059.jpg: 0.9975 объектов: 35
stack-old-red-bricks-text

## Предобученная модель yolov8s

### Обучение

In [20]:
pretrained_weights = './train_result/yolov8s_training/train/weights/best.pt'
project_name = "yolov8s_training_transfer_learning"

In [21]:
results = train_two_stage_yolo(
    data_yaml=DATA_YAML,
    project_name=project_name,
    pretrained_weights=pretrained_weights,

    output_dir=RESULTS_DIR,

    imgsz=640,
    batch=BATCH_SIZE,

    stage1_epochs=200,
    stage1_freeze=10,
    stage1_lr=1e-4,

    stage2_epochs=200,
    stage2_lr=1e-5,

    patience=PATIENCE,

    device=DEVICE
)


STAGE 1: TRAIN HEAD WITH FROZEN BACKBONE
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./new_block_detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.

### Визуализация на новом наборе данных

In [22]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

new_block_detection-2/valid/images
Обработка 1/11: vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  Сохранено: comparison_001_vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  GT: 5 объектов, Predictions: 15 объектов

Обработка 2/11: vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  Сохранено: comparison_002_vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  GT: 36 объектов, Predictions: 64 объектов

Обработка 3/11: vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  Сохранено: comparison_003_vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  GT: 18 объектов, Predictions: 32 объектов

Обработка 4/11: vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  Сохранено: comparison_004_vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  GT: 28 объектов, Predictions: 57 объектов

Обработка 5/11: vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg
  Сохранено: comparison_005_vid0499

'./train_result_new_data/yolov8s_training_transfer_learning/val/res'

### Визуализация на старом наборе данных

In [23]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    './block-detection-1/data.yaml',
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/123: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/123: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 36 объектов

Обработка 3/123: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 40 объектов

Обработка 4/123: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/123: _barhat-2-500x500-500x500_jp

  Сохранено: comparison_005__barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg
  GT: 28 объектов, Predictions: 27 объектов

Обработка 6/123: stack-red-bricks-background-86570305_jpg.rf.c237a2a5e4ce181a7aaaebae7bbc358d.jpg
  Сохранено: comparison_006_stack-red-bricks-background-86570305_jpg.rf.c237a2a5e4ce181a7aaaebae7bbc358d.jpg
  GT: 49 объектов, Predictions: 19 объектов

Обработка 7/123: 02-500x500-1-_jpg.rf.0c6c8f9c50bb37fb3fe32d5f9686329a.jpg
  Сохранено: comparison_007_02-500x500-1-_jpg.rf.0c6c8f9c50bb37fb3fe32d5f9686329a.jpg
  GT: 143 объектов, Predictions: 145 объектов

Обработка 8/123: stacked-orange-bricks-construction-site-stacked-orange-bricks-construction-site-brick-wall-184256617_jpg.rf.1dbe7e1e4e1a5f2fd11ef1137ffa3113.jpg
  Сохранено: comparison_008_stacked-orange-bricks-construction-site-stacked-orange-bricks-construction-site-brick-wall-184256617_jpg.rf.1dbe7e1e4e1a5f2fd11ef1137ffa3113.jpg
  GT: 85 объектов, Predictions: 103 объектов

Обработка 9/123:

'./train_result_new_data/yolov8s_training_transfer_learning/val/res'

### Валидация на старом наборе данных

In [24]:
validate_model('./train_result_new_data/yolov8s_training_transfer_learning/stage2/train/weights/best.pt',
               './block-detection-1/data.yaml')


VALIDATING: ./train_result_new_data/yolov8s_training_transfer_learning/stage2/train/weights/best.pt

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2757.5±542.6 MB/s, size: 76.7 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 30.3Mit/s 0.0s


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.9it/s 4.2s0.3ss
                   all        123       8279      0.808       0.79      0.811      0.367
Speed: 1.7ms preprocess, 16.9ms inference, 0.3ms loss, 7.0ms postprocess per image
Results saved to /home/himoza/block_detect/runs/detect/val-39

Validation Metrics:
mAP50: 0.8106
mAP50-95: 0.3667
precision: 0.8082
recall: 0.7902


{'mAP50': np.float64(0.8105753245297577),
 'mAP50-95': np.float64(0.3666607179689331),
 'precision': np.float64(0.8081570113704742),
 'recall': np.float64(0.7902115767524003)}

### Валидация на нашей метрике Intersection / Ground Truth на новом наборе данных

In [25]:
model = YOLO("./train_result_new_data/yolov8s_training_transfer_learning/best_final_model.pt")

images_dir = Path("./new_block_detection-2/valid/images")
labels_dir = Path("./new_block_detection-2/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg: 0.9789 объектов: 36
vid0493img330_3_2_jpg.rf.9b32fb16517572340e510438f4a2b700.jpg: 0.9385 объектов: 27
vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg: 0.9136 объектов: 10
vid0499img1380_4_1_jpg.rf.7ac56021fb5d32f299ee02e2f3f942e4.jpg: 0.8284 объектов: 8
vid0489img2280_1_2_jpg.rf.3f1bd4eae4d057439a2f91baf1de3333.jpg: 0.9463 объектов: 36
vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg: 0.7413 объектов: 5
vid0498img1110_4_2_jpg.rf.655393520c50bf01898d6ed9653a076c.jpg: 0.9780 объектов: 15
vid0498img30_4_2_jpg.rf.285af3e44d3400585ee7668aace70aad.jpg: 0.9182 объектов: 3
vid0489img2250_2_1_jpg.rf.06985a295a14e641f58c90652bc17b55.jpg: 0.9810 объектов: 32
vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg: 0.9761 объектов: 18
vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg: 0.9766 объектов: 28

Intersection / GT: 0.9535
Количество объектов: 218


### Валидация на нашей метрике Intersection / Ground Truth на старом наборе данных

In [26]:
model = YOLO("./train_result_new_data/yolov8s_training_transfer_learning/best_final_model.pt")

images_dir = Path("./block-detection-1/valid/images")
labels_dir = Path("./block-detection-1/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

many-stack-brick-background-construction-site-69549698_jpg.rf.2c1d4a0251603f35b6e7241f23e36a28.jpg: 0.9955 объектов: 114
vandersanden-kirpich-ruchnoy-formovki-oud-kapelle-m50-190-90-50-1_0001_w500_h500_watermark-2-_jpg.rf.8c61c216f6a48614049e8bea63053d13.jpg: 0.9949 объектов: 48
stack-bricks-red-construction-36132343_jpg.rf.a3ab585f8df68d8aa172d0b98a9d0ab1.jpg: 0.9858 объектов: 45
brick2-17-_jpg.rf.f64be790718963275d995fa4b939e546.jpg: 0.9474 объектов: 81
_barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg: 0.9993 объектов: 27
383_jpg.rf.bbc945440a6bb651e33008744c19bcd2.jpg: 0.8749 объектов: 27
4-1-_jpg.rf.38692d9e8160c842930d08b826f87182.jpg: 0.9967 объектов: 104
full_4404479-0-1-_jpg.rf.3fffd358b0471fd970c05ddeac0945e5.jpg: 0.9684 объектов: 41
stack-bricks-construction-territory-near-future-house-72727138_jpg.rf.4dcad82f85499f905a9cf3aa286f0c44.jpg: 0.9884 объектов: 63
unnamed_jpg.rf.2843209a631061453c3a09b086990059.jpg: 0.9951 объектов: 35
stack-old-red-bricks-text

## Предобученная модель yolov11n

### Обучение

In [27]:
pretrained_weights = './train_result/yolov11n_training/train/weights/best.pt'
project_name = "yolov11n_training_transfer_learning"

In [28]:
results = train_two_stage_yolo(
    data_yaml=DATA_YAML,
    project_name=project_name,
    pretrained_weights=pretrained_weights,

    output_dir=RESULTS_DIR,

    imgsz=640,
    batch=BATCH_SIZE,

    stage1_epochs=200,
    stage1_freeze=10,
    stage1_lr=1e-4,

    stage2_epochs=200,
    stage2_lr=1e-5,

    patience=PATIENCE,

    device=DEVICE
)


STAGE 1: TRAIN HEAD WITH FROZEN BACKBONE
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./new_block_detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.

### Визуализация на новом наборе данных

In [29]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

new_block_detection-2/valid/images
Обработка 1/11: vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  Сохранено: comparison_001_vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  GT: 5 объектов, Predictions: 14 объектов

Обработка 2/11: vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  Сохранено: comparison_002_vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  GT: 36 объектов, Predictions: 59 объектов

Обработка 3/11: vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  Сохранено: comparison_003_vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  GT: 18 объектов, Predictions: 29 объектов

Обработка 4/11: vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  Сохранено: comparison_004_vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  GT: 28 объектов, Predictions: 50 объектов

Обработка 5/11: vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg
  Сохранено: comparison_005_vid0499

'./train_result_new_data/yolov11n_training_transfer_learning/val/res'

### Визуализация на старом наборе данных

In [30]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    './block-detection-1/data.yaml',
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/123: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg


  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 23 объектов

Обработка 2/123: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 33 объектов

Обработка 3/123: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 33 объектов

Обработка 4/123: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/123: _barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg
  Сохранено: comparison_005__barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a7545

'./train_result_new_data/yolov11n_training_transfer_learning/val/res'

### Валидация на старом наборе данных

In [31]:
validate_model('./train_result_new_data/yolov11n_training_transfer_learning/stage2/train/weights/best.pt',
               './block-detection-1/data.yaml')


VALIDATING: ./train_result_new_data/yolov11n_training_transfer_learning/stage2/train/weights/best.pt



Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2097.8±878.0 MB/s, size: 68.0 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 86.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.0it/s 3.9s0.2ss
                   all        123       8279      0.868      0.852      0.886      0.444
Speed: 2.1ms preprocess, 13.2ms inference, 0.0ms loss, 7.6ms postprocess per image
Results saved to /home/himoza/block_detect/runs/detect/val-42

Validation Metrics:
mAP50: 0.8863
mAP50-95: 0.4442
precision: 0.8681
recall: 0.8522


{'mAP50': np.float64(0.886276938284062),
 'mAP50-95': np.float64(0.4442383862922492),
 'precision': np.float64(0.8681022480566734),
 'recall': np.float64(0.8521560574948666)}

### Валидация на нашей метрике Intersection / Ground Truth на новом наборе данных

In [32]:
model = YOLO("./train_result_new_data/yolov11n_training_transfer_learning/best_final_model.pt")

images_dir = Path("./new_block_detection-2/valid/images")
labels_dir = Path("./new_block_detection-2/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg: 0.9741 объектов: 36
vid0493img330_3_2_jpg.rf.9b32fb16517572340e510438f4a2b700.jpg: 0.9393 объектов: 27
vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg: 0.9361 объектов: 10
vid0499img1380_4_1_jpg.rf.7ac56021fb5d32f299ee02e2f3f942e4.jpg: 0.8093 объектов: 8
vid0489img2280_1_2_jpg.rf.3f1bd4eae4d057439a2f91baf1de3333.jpg: 0.9450 объектов: 36
vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg: 0.8063 объектов: 5
vid0498img1110_4_2_jpg.rf.655393520c50bf01898d6ed9653a076c.jpg: 0.9778 объектов: 15
vid0498img30_4_2_jpg.rf.285af3e44d3400585ee7668aace70aad.jpg: 0.8941 объектов: 2
vid0489img2250_2_1_jpg.rf.06985a295a14e641f58c90652bc17b55.jpg: 0.9769 объектов: 32
vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg: 0.9730 объектов: 18
vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg: 0.9757 объектов: 28

Intersection / GT: 0.9533
Количество объектов: 217


### Валидация на нашей метрике Intersection / Ground Truth на старом наборе данных

In [33]:
model = YOLO("./train_result_new_data/yolov11n_training_transfer_learning/best_final_model.pt")

images_dir = Path("./block-detection-1/valid/images")
labels_dir = Path("./block-detection-1/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

many-stack-brick-background-construction-site-69549698_jpg.rf.2c1d4a0251603f35b6e7241f23e36a28.jpg: 0.9961 объектов: 142
vandersanden-kirpich-ruchnoy-formovki-oud-kapelle-m50-190-90-50-1_0001_w500_h500_watermark-2-_jpg.rf.8c61c216f6a48614049e8bea63053d13.jpg: 0.9977 объектов: 48
stack-bricks-red-construction-36132343_jpg.rf.a3ab585f8df68d8aa172d0b98a9d0ab1.jpg: 0.9836 объектов: 44
brick2-17-_jpg.rf.f64be790718963275d995fa4b939e546.jpg: 0.9906 объектов: 86
_barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg: 0.9989 объектов: 24
383_jpg.rf.bbc945440a6bb651e33008744c19bcd2.jpg: 0.9244 объектов: 23
4-1-_jpg.rf.38692d9e8160c842930d08b826f87182.jpg: 0.9936 объектов: 104
full_4404479-0-1-_jpg.rf.3fffd358b0471fd970c05ddeac0945e5.jpg: 0.9885 объектов: 42
stack-bricks-construction-territory-near-future-house-72727138_jpg.rf.4dcad82f85499f905a9cf3aa286f0c44.jpg: 0.9603 объектов: 72
unnamed_jpg.rf.2843209a631061453c3a09b086990059.jpg: 0.9966 объектов: 31
stack-old-red-bricks-text

## Предобученная модель yolov11s

### Обучение

In [13]:
pretrained_weights = './train_result/yolov11s_training/train/weights/best.pt'
project_name = "yolov11s_training_transfer_learning"

In [14]:
results = train_two_stage_yolo(
    data_yaml=DATA_YAML,
    project_name=project_name,
    pretrained_weights=pretrained_weights,

    output_dir=RESULTS_DIR,

    imgsz=640,
    batch=BATCH_SIZE,

    stage1_epochs=200,
    stage1_freeze=10,
    stage1_lr=1e-4,

    stage2_epochs=200,
    stage2_lr=1e-5,

    patience=PATIENCE,

    device=DEVICE
)


STAGE 1: TRAIN HEAD WITH FROZEN BACKBONE
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./new_block_detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.

### Визуализация на новом наборе данных

In [15]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

new_block_detection-2/valid/images
Обработка 1/11: vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  Сохранено: comparison_001_vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  GT: 5 объектов, Predictions: 18 объектов

Обработка 2/11: vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  Сохранено: comparison_002_vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  GT: 36 объектов, Predictions: 56 объектов

Обработка 3/11: vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  Сохранено: comparison_003_vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  GT: 18 объектов, Predictions: 28 объектов

Обработка 4/11: vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  Сохранено: comparison_004_vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  GT: 28 объектов, Predictions: 53 объектов

Обработка 5/11: vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg
  Сохранено: comparison_005_vid0499

'./train_result_new_data/yolov11s_training_transfer_learning/val/res'

### Визуализация на старом наборе данных

In [16]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    './block-detection-1/data.yaml',
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/123: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/123: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 36 объектов

Обработка 3/123: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 40 объектов

Обработка 4/123: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/123: _barhat-2-500x500-500x500_jp

  Сохранено: comparison_022_artworks-000043721612-9cj65t-t500x500_jpg.rf.3f646912d04a0ef332997163613daa42.jpg
  GT: 87 объектов, Predictions: 88 объектов

Обработка 23/123: UK_MKT_12716010_Whitby_Red_Multi_Rustica_jpg.rf.2ff91ae8931d26b8c5a1f520650fe914.jpg
  Сохранено: comparison_023_UK_MKT_12716010_Whitby_Red_Multi_Rustica_jpg.rf.2ff91ae8931d26b8c5a1f520650fe914.jpg
  GT: 24 объектов, Predictions: 20 объектов

Обработка 24/123: white-bricks-stack-construction-site-43235571_jpg.rf.c770bdd25444a950899e9f64f37dc214.jpg
  Сохранено: comparison_024_white-bricks-stack-construction-site-43235571_jpg.rf.c770bdd25444a950899e9f64f37dc214.jpg
  GT: 151 объектов, Predictions: 148 объектов

Обработка 25/123: stacks-bricks-3106628_jpg.rf.f9f991dbb87ce6c03d8e3faae63cc4de.jpg
  Сохранено: comparison_025_stacks-bricks-3106628_jpg.rf.f9f991dbb87ce6c03d8e3faae63cc4de.jpg
  GT: 95 объектов, Predictions: 81 объектов

Обработка 26/123: UK_MKT_24250080_Chartham_Multi_Stock_jpg.rf.9433c0c83bb13c19e179bcd4af

'./train_result_new_data/yolov11s_training_transfer_learning/val/res'

### Валидация на старом наборе данных

In [17]:
validate_model('./train_result_new_data/yolov11s_training_transfer_learning/stage2/train/weights/best.pt',
               './block-detection-1/data.yaml')


VALIDATING: ./train_result_new_data/yolov11s_training_transfer_learning/stage2/train/weights/best.pt

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2792.5±635.3 MB/s, size: 76.7 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 73.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.8it/s 4.4s0.3ss
                   all        123       8279      0.866      0.846      0.881       0.42
Speed: 2.2ms preprocess, 17.1ms inference, 0.0ms loss, 6.9ms postprocess per image
Results saved to /home/himoza/block_detect/runs/detect/val-45

Validation Metrics:
mAP50: 0.8809
mAP50-95: 0.4205
precision: 0.8663
recall: 0.8461


{'mAP50': np.float64(0.8808591645532937),
 'mAP50-95': np.float64(0.4204978483760956),
 'precision': np.float64(0.8662547197152939),
 'recall': np.float64(0.8461166807585457)}

### Валидация на нашей метрике Intersection / Ground Truth на новом наборе данных

In [18]:
model = YOLO("./train_result_new_data/yolov11s_training_transfer_learning/best_final_model.pt")

images_dir = Path("./new_block_detection-2/valid/images")
labels_dir = Path("./new_block_detection-2/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg: 0.9790 объектов: 36
vid0493img330_3_2_jpg.rf.9b32fb16517572340e510438f4a2b700.jpg: 0.9460 объектов: 27
vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg: 0.9283 объектов: 10
vid0499img1380_4_1_jpg.rf.7ac56021fb5d32f299ee02e2f3f942e4.jpg: 0.7789 объектов: 8
vid0489img2280_1_2_jpg.rf.3f1bd4eae4d057439a2f91baf1de3333.jpg: 0.9447 объектов: 36
vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg: 0.6847 объектов: 5
vid0498img1110_4_2_jpg.rf.655393520c50bf01898d6ed9653a076c.jpg: 0.9825 объектов: 15
vid0498img30_4_2_jpg.rf.285af3e44d3400585ee7668aace70aad.jpg: 0.9032 объектов: 3
vid0489img2250_2_1_jpg.rf.06985a295a14e641f58c90652bc17b55.jpg: 0.9782 объектов: 32
vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg: 0.9736 объектов: 18
vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg: 0.9804 объектов: 28

Intersection / GT: 0.9517
Количество объектов: 218


### Валидация на нашей метрике Intersection / Ground Truth на старом наборе данных

In [19]:
model = YOLO("./train_result_new_data/yolov11s_training_transfer_learning/best_final_model.pt")

images_dir = Path("./block-detection-1/valid/images")
labels_dir = Path("./block-detection-1/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

many-stack-brick-background-construction-site-69549698_jpg.rf.2c1d4a0251603f35b6e7241f23e36a28.jpg: 0.9930 объектов: 141
vandersanden-kirpich-ruchnoy-formovki-oud-kapelle-m50-190-90-50-1_0001_w500_h500_watermark-2-_jpg.rf.8c61c216f6a48614049e8bea63053d13.jpg: 0.9971 объектов: 48
stack-bricks-red-construction-36132343_jpg.rf.a3ab585f8df68d8aa172d0b98a9d0ab1.jpg: 0.9686 объектов: 46
brick2-17-_jpg.rf.f64be790718963275d995fa4b939e546.jpg: 0.9792 объектов: 86
_barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg: 0.9993 объектов: 25
383_jpg.rf.bbc945440a6bb651e33008744c19bcd2.jpg: 0.8881 объектов: 27
4-1-_jpg.rf.38692d9e8160c842930d08b826f87182.jpg: 0.9950 объектов: 104
full_4404479-0-1-_jpg.rf.3fffd358b0471fd970c05ddeac0945e5.jpg: 0.9926 объектов: 42
stack-bricks-construction-territory-near-future-house-72727138_jpg.rf.4dcad82f85499f905a9cf3aa286f0c44.jpg: 0.9858 объектов: 73
unnamed_jpg.rf.2843209a631061453c3a09b086990059.jpg: 0.9996 объектов: 35
stack-old-red-bricks-text

## Предобученная модель yolov26n

### Обучение

In [20]:
pretrained_weights = './train_result/aug_yolo26n_training/train/weights/best.pt'
project_name = "yolov26n_training_transfer_learning"

In [21]:
results = train_two_stage_yolo(
    data_yaml=DATA_YAML,
    project_name=project_name,
    pretrained_weights=pretrained_weights,

    output_dir=RESULTS_DIR,

    imgsz=640,
    batch=BATCH_SIZE,

    stage1_epochs=200,
    stage1_freeze=10,
    stage1_lr=1e-4,

    stage2_epochs=200,
    stage2_lr=1e-5,

    patience=PATIENCE,

    device=DEVICE
)


STAGE 1: TRAIN HEAD WITH FROZEN BACKBONE
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./new_block_detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.

### Визуализация на новом наборе данных

In [22]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

new_block_detection-2/valid/images
Обработка 1/11: vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  Сохранено: comparison_001_vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  GT: 5 объектов, Predictions: 13 объектов

Обработка 2/11: vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  Сохранено: comparison_002_vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  GT: 36 объектов, Predictions: 59 объектов

Обработка 3/11: vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  Сохранено: comparison_003_vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  GT: 18 объектов, Predictions: 31 объектов

Обработка 4/11: vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  Сохранено: comparison_004_vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  GT: 28 объектов, Predictions: 52 объектов

Обработка 5/11: vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg
  Сохранено: comparison_005_vid0499

'./train_result_new_data/yolov26n_training_transfer_learning/val/res'

### Визуализация на старом наборе данных

In [23]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    './block-detection-1/data.yaml',
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/123: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg


  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 23 объектов

Обработка 2/123: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 28 объектов

Обработка 3/123: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 37 объектов

Обработка 4/123: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 91 объектов

Обработка 5/123: _barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg
  Сохранено: comparison_005__barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454

'./train_result_new_data/yolov26n_training_transfer_learning/val/res'

### Валидация на старом наборе данных

In [24]:
validate_model('./train_result_new_data/yolov26n_training_transfer_learning/stage2/train/weights/best.pt',
               './block-detection-1/data.yaml')


VALIDATING: ./train_result_new_data/yolov26n_training_transfer_learning/stage2/train/weights/best.pt

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
YOLO26n summary (fused): 120 layers, 2,375,031 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2541.9±326.3 MB/s, size: 68.8 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 51.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.9it/s 4.1s0.2ss
                   all        123       8279        0.9      0.865       0.92      0.379
Speed: 2.1ms preprocess, 20.2ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /home/himoza/block_detect/runs/detect/val-48

Validation Metrics:
mAP50: 0.9201
mAP50-95: 0.3792
precision: 0.9003
recall: 0.8649


{'mAP50': np.float64(0.920101535173697),
 'mAP50-95': np.float64(0.37917865341905327),
 'precision': np.float64(0.9002974506778866),
 'recall': np.float64(0.8649182556526451)}

### Валидация на нашей метрике Intersection / Ground Truth на новом наборе данных

In [25]:
model = YOLO("./train_result_new_data/yolov26n_training_transfer_learning/best_final_model.pt")

images_dir = Path("./new_block_detection-2/valid/images")
labels_dir = Path("./new_block_detection-2/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg: 0.9817 объектов: 36
vid0493img330_3_2_jpg.rf.9b32fb16517572340e510438f4a2b700.jpg: 0.9385 объектов: 27
vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg: 0.9164 объектов: 10
vid0499img1380_4_1_jpg.rf.7ac56021fb5d32f299ee02e2f3f942e4.jpg: 0.8268 объектов: 8
vid0489img2280_1_2_jpg.rf.3f1bd4eae4d057439a2f91baf1de3333.jpg: 0.9521 объектов: 36
vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg: 0.7445 объектов: 5
vid0498img1110_4_2_jpg.rf.655393520c50bf01898d6ed9653a076c.jpg: 0.9786 объектов: 15
vid0498img30_4_2_jpg.rf.285af3e44d3400585ee7668aace70aad.jpg: 0.8953 объектов: 3
vid0489img2250_2_1_jpg.rf.06985a295a14e641f58c90652bc17b55.jpg: 0.9812 объектов: 32
vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg: 0.9751 объектов: 18
vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg: 0.9780 объектов: 28

Intersection / GT: 0.9548
Количество объектов: 218


### Валидация на нашей метрике Intersection / Ground Truth на старом наборе данных

In [26]:
model = YOLO("./train_result_new_data/yolov26n_training_transfer_learning/best_final_model.pt")

images_dir = Path("./block-detection-1/valid/images")
labels_dir = Path("./block-detection-1/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

many-stack-brick-background-construction-site-69549698_jpg.rf.2c1d4a0251603f35b6e7241f23e36a28.jpg: 0.9999 объектов: 125
vandersanden-kirpich-ruchnoy-formovki-oud-kapelle-m50-190-90-50-1_0001_w500_h500_watermark-2-_jpg.rf.8c61c216f6a48614049e8bea63053d13.jpg: 0.9980 объектов: 39
stack-bricks-red-construction-36132343_jpg.rf.a3ab585f8df68d8aa172d0b98a9d0ab1.jpg: 0.9612 объектов: 39
brick2-17-_jpg.rf.f64be790718963275d995fa4b939e546.jpg: 0.9870 объектов: 74
_barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg: 0.9982 объектов: 27
383_jpg.rf.bbc945440a6bb651e33008744c19bcd2.jpg: 0.8946 объектов: 26
4-1-_jpg.rf.38692d9e8160c842930d08b826f87182.jpg: 0.9953 объектов: 104
full_4404479-0-1-_jpg.rf.3fffd358b0471fd970c05ddeac0945e5.jpg: 0.9915 объектов: 41
stack-bricks-construction-territory-near-future-house-72727138_jpg.rf.4dcad82f85499f905a9cf3aa286f0c44.jpg: 0.9838 объектов: 51
unnamed_jpg.rf.2843209a631061453c3a09b086990059.jpg: 0.9973 объектов: 35
stack-old-red-bricks-text

## Предобученная модель yolov26s

### Обучение

In [13]:
pretrained_weights = './train_result/aug_yolo26s_training/train/weights/best.pt'
project_name = "yolov26s_training_transfer_learning"

In [14]:
results = train_two_stage_yolo(
    data_yaml=DATA_YAML,
    project_name=project_name,
    pretrained_weights=pretrained_weights,

    output_dir=RESULTS_DIR,

    imgsz=640,
    batch=BATCH_SIZE,

    stage1_epochs=200,
    stage1_freeze=10,
    stage1_lr=1e-4,

    stage2_epochs=200,
    stage2_lr=1e-5,

    patience=PATIENCE,

    device=DEVICE
)


STAGE 1: TRAIN HEAD WITH FROZEN BACKBONE
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./new_block_detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.

### Визуализация на новом наборе данных

In [15]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

new_block_detection-2/valid/images
Обработка 1/11: vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  Сохранено: comparison_001_vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  GT: 5 объектов, Predictions: 15 объектов

Обработка 2/11: vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  Сохранено: comparison_002_vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  GT: 36 объектов, Predictions: 55 объектов

Обработка 3/11: vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  Сохранено: comparison_003_vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  GT: 18 объектов, Predictions: 24 объектов

Обработка 4/11: vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  Сохранено: comparison_004_vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  GT: 28 объектов, Predictions: 46 объектов

Обработка 5/11: vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg
  Сохранено: comparison_005_vid0499

'./train_result_new_data/yolov26s_training_transfer_learning/val/res'

### Визуализация на старом наборе данных

In [16]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    './block-detection-1/data.yaml',
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/123: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg


  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/123: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 31 объектов

Обработка 3/123: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 39 объектов

Обработка 4/123: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 93 объектов

Обработка 5/123: _barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg
  Сохранено: comparison_005__barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454

'./train_result_new_data/yolov26s_training_transfer_learning/val/res'

### Валидация на старом наборе данных

In [17]:
validate_model('./train_result_new_data/yolov26s_training_transfer_learning/stage2/train/weights/best.pt',
               './block-detection-1/data.yaml')


VALIDATING: ./train_result_new_data/yolov26s_training_transfer_learning/stage2/train/weights/best.pt

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)


YOLO26s summary (fused): 120 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2598.0±712.0 MB/s, size: 77.7 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 73.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.8it/s 4.4s0.3ss
                   all        123       8279      0.895      0.836      0.897      0.383
Speed: 2.1ms preprocess, 17.4ms inference, 0.0ms loss, 6.6ms postprocess per image
Results saved to /home/himoza/block_detect/runs/detect/val-51

Validation Metrics:
mAP50: 0.8965
mAP50-95: 0.3829
precision: 0.8948
recall: 0.8356


{'mAP50': np.float64(0.8965303598466801),
 'mAP50-95': np.float64(0.3829281975279756),
 'precision': np.float64(0.8948454320685525),
 'recall': np.float64(0.8356081652373475)}

### Валидация на нашей метрике Intersection / Ground Truth на новом наборе данных

In [18]:
model = YOLO("./train_result_new_data/yolov26s_training_transfer_learning/best_final_model.pt")

images_dir = Path("./new_block_detection-2/valid/images")
labels_dir = Path("./new_block_detection-2/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg: 0.9766 объектов: 36
vid0493img330_3_2_jpg.rf.9b32fb16517572340e510438f4a2b700.jpg: 0.9473 объектов: 27
vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg: 0.9394 объектов: 10
vid0499img1380_4_1_jpg.rf.7ac56021fb5d32f299ee02e2f3f942e4.jpg: 0.8073 объектов: 7
vid0489img2280_1_2_jpg.rf.3f1bd4eae4d057439a2f91baf1de3333.jpg: 0.9498 объектов: 36
vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg: 0.7337 объектов: 5
vid0498img1110_4_2_jpg.rf.655393520c50bf01898d6ed9653a076c.jpg: 0.9811 объектов: 15
vid0498img30_4_2_jpg.rf.285af3e44d3400585ee7668aace70aad.jpg: 0.9520 объектов: 3
vid0489img2250_2_1_jpg.rf.06985a295a14e641f58c90652bc17b55.jpg: 0.9852 объектов: 32
vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg: 0.9740 объектов: 18
vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg: 0.9809 объектов: 28

Intersection / GT: 0.9573
Количество объектов: 217


### Валидация на нашей метрике Intersection / Ground Truth на старом наборе данных

In [19]:
model = YOLO("./train_result_new_data/yolov26s_training_transfer_learning/best_final_model.pt")

images_dir = Path("./block-detection-1/valid/images")
labels_dir = Path("./block-detection-1/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

many-stack-brick-background-construction-site-69549698_jpg.rf.2c1d4a0251603f35b6e7241f23e36a28.jpg: 0.9988 объектов: 125
vandersanden-kirpich-ruchnoy-formovki-oud-kapelle-m50-190-90-50-1_0001_w500_h500_watermark-2-_jpg.rf.8c61c216f6a48614049e8bea63053d13.jpg: 0.9993 объектов: 42
stack-bricks-red-construction-36132343_jpg.rf.a3ab585f8df68d8aa172d0b98a9d0ab1.jpg: 0.9964 объектов: 42
brick2-17-_jpg.rf.f64be790718963275d995fa4b939e546.jpg: 0.9945 объектов: 83
_barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg: 0.9938 объектов: 22
383_jpg.rf.bbc945440a6bb651e33008744c19bcd2.jpg: 0.9434 объектов: 23
4-1-_jpg.rf.38692d9e8160c842930d08b826f87182.jpg: 0.9976 объектов: 103
full_4404479-0-1-_jpg.rf.3fffd358b0471fd970c05ddeac0945e5.jpg: 0.9964 объектов: 42
stack-bricks-construction-territory-near-future-house-72727138_jpg.rf.4dcad82f85499f905a9cf3aa286f0c44.jpg: 0.9919 объектов: 53
unnamed_jpg.rf.2843209a631061453c3a09b086990059.jpg: 0.9987 объектов: 32
stack-old-red-bricks-text

## Предобученная модель yolov26m

### Обучение

In [13]:
pretrained_weights = './train_result/aug_yolo26m_training/train/weights/best.pt'
project_name = "yolov26m_training_transfer_learning"

In [14]:
results = train_two_stage_yolo(
    data_yaml=DATA_YAML,
    project_name=project_name,
    pretrained_weights=pretrained_weights,

    output_dir=RESULTS_DIR,

    imgsz=640,
    batch=16,

    stage1_epochs=200,
    stage1_freeze=10,
    stage1_lr=1e-4,

    stage2_epochs=200,
    stage2_lr=1e-5,

    patience=PATIENCE,

    device=DEVICE
)


STAGE 1: TRAIN HEAD WITH FROZEN BACKBONE


New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=None, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=./new_block_detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, m

### Визуализация на новом наборе данных

In [15]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    DATA_YAML,
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

new_block_detection-2/valid/images
Обработка 1/11: vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  Сохранено: comparison_001_vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg
  GT: 5 объектов, Predictions: 12 объектов

Обработка 2/11: vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  Сохранено: comparison_002_vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg
  GT: 36 объектов, Predictions: 46 объектов

Обработка 3/11: vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  Сохранено: comparison_003_vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg
  GT: 18 объектов, Predictions: 24 объектов

Обработка 4/11: vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  Сохранено: comparison_004_vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg
  GT: 28 объектов, Predictions: 38 объектов

Обработка 5/11: vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg
  Сохранено: comparison_005_vid0499

'./train_result_new_data/yolov26m_training_transfer_learning/val/res'

### Визуализация на старом наборе данных

In [16]:
visualize_predictions_vs_gt(
    project_name,
    RESULTS_DIR,
    './block-detection-1/data.yaml',
    num_samples=400,
    conf_threshold=0.3,
    iou_threshold=0.4,
    img_size=640,
    seed=42
)

block-detection-1/valid/images
Обработка 1/123: 6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  Сохранено: comparison_001_6fbc3bab88742020120b48e272915d25_jpg.rf.51c474a8809d18bf99787cd15a510aaf.jpg
  GT: 24 объектов, Predictions: 24 объектов

Обработка 2/123: artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  Сохранено: comparison_002_artworks-000022313743-yn12zz-t500x500_jpg.rf.e28a4b4000ee8c019097a5f70d977bd5.jpg
  GT: 36 объектов, Predictions: 30 объектов

Обработка 3/123: wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  Сохранено: comparison_003_wall-brick-red-brick-500x500-2-_jpg.rf.75939b79af1bc9181475619f8f737cdd.jpg
  GT: 40 объектов, Predictions: 37 объектов

Обработка 4/123: 4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  Сохранено: comparison_004_4_jpg.rf.7e70007b71984be56478ff363b8d6906.jpg
  GT: 104 объектов, Predictions: 104 объектов

Обработка 5/123: _barhat-2-500x500-500x500_jp

'./train_result_new_data/yolov26m_training_transfer_learning/val/res'

### Валидация на старом наборе данных

In [17]:
validate_model('./train_result_new_data/yolov26m_training_transfer_learning/stage2/train/weights/best.pt',
               './block-detection-1/data.yaml')


VALIDATING: ./train_result_new_data/yolov26m_training_transfer_learning/stage2/train/weights/best.pt

Ultralytics 8.4.168 🚀 Python-3.13.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060, 5730MiB)
YOLO26m summary (fused): 130 layers, 20,350,223 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2085.4±498.9 MB/s, size: 88.1 KB)
val: Scanning /home/himoza/block_detect/block-detection-1/valid/labels.cache... 123 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 123/123 73.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.5it/s 5.5s0.5ss
                   all        123       8279      0.879      0.804       0.87      0.323
Speed: 2.2ms preprocess, 31.8ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /home/himoza/block_detect/runs/detect/val-54

Validation Metrics:
mAP50: 0.8705
mAP50-95: 0.3226
precision: 0.8792
recall: 0.8037


{'mAP50': np.float64(0.8704847953160069),
 'mAP50-95': np.float64(0.3225812690199796),
 'precision': np.float64(0.879220995083467),
 'recall': np.float64(0.803664746905265)}

### Валидация на нашей метрике Intersection / Ground Truth на новом наборе данных

In [18]:
model = YOLO("./train_result_new_data/yolov26m_training_transfer_learning/best_final_model.pt")

images_dir = Path("./new_block_detection-2/valid/images")
labels_dir = Path("./new_block_detection-2/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

vid0489img30_4_1_jpg.rf.aca42f6e41026a1b4bd32ef6a78d28c5.jpg: 0.9823 объектов: 36
vid0493img330_3_2_jpg.rf.9b32fb16517572340e510438f4a2b700.jpg: 0.9404 объектов: 27
vid0499img1200_3_0_jpg.rf.9972bb714676db1a9d62d8f25325476b.jpg: 0.9276 объектов: 10
vid0499img1380_4_1_jpg.rf.7ac56021fb5d32f299ee02e2f3f942e4.jpg: 0.8159 объектов: 8
vid0489img2280_1_2_jpg.rf.3f1bd4eae4d057439a2f91baf1de3333.jpg: 0.9441 объектов: 36
vid0499img1290_1_0_jpg.rf.59b9909ce55a9c1e8168e2d1ee2692f1.jpg: 0.7342 объектов: 5
vid0498img1110_4_2_jpg.rf.655393520c50bf01898d6ed9653a076c.jpg: 0.9868 объектов: 15
vid0498img30_4_2_jpg.rf.285af3e44d3400585ee7668aace70aad.jpg: 0.8779 объектов: 3
vid0489img2250_2_1_jpg.rf.06985a295a14e641f58c90652bc17b55.jpg: 0.9802 объектов: 32
vid0489img2250_4_0_jpg.rf.453d246b437d688264d264b99e30a23b.jpg: 0.9715 объектов: 18
vid0489img30_4_0_jpg.rf.5c211fe4579a2f8d27f91d5753fb1398.jpg: 0.9828 объектов: 27

Intersection / GT: 0.9541
Количество объектов: 217


### Валидация на нашей метрике Intersection / Ground Truth на старом наборе данных

In [19]:
model = YOLO("./train_result_new_data/yolov26m_training_transfer_learning/best_final_model.pt")

images_dir = Path("./block-detection-1/valid/images")
labels_dir = Path("./block-detection-1/valid/labels")

all_scores = []

for image_path in images_dir.glob("*"):

    label_path = labels_dir / f"{image_path.stem}.txt"

    image_metric, matches = evaluate_image(
        model,
        image_path,
        label_path,
        matching_iou_threshold=0.3
    )

    # Если на изображении не было ни одного
    # сопоставленного объекта — пропускаем его
    if image_metric is None:
        continue

    all_scores.extend(
        match["intersection_gt"]
        for match in matches
    )

    print(
        f"{image_path.name}: "
        f"{image_metric:.4f} "
        f"объектов: {len(matches)}"
    )


# --------------------------------
# Общая метрика
# --------------------------------

if len(all_scores) > 0:

    final_metric = np.mean(all_scores)

    print("\n==============================")
    print(
        f"Intersection / GT: "
        f"{final_metric:.4f}"
    )
    print(
        f"Количество объектов: "
        f"{len(all_scores)}"
    )
    print("==============================")

else:

    print("Нет сопоставленных объектов.")

many-stack-brick-background-construction-site-69549698_jpg.rf.2c1d4a0251603f35b6e7241f23e36a28.jpg: 0.9999 объектов: 130
vandersanden-kirpich-ruchnoy-formovki-oud-kapelle-m50-190-90-50-1_0001_w500_h500_watermark-2-_jpg.rf.8c61c216f6a48614049e8bea63053d13.jpg: 0.9980 объектов: 48
stack-bricks-red-construction-36132343_jpg.rf.a3ab585f8df68d8aa172d0b98a9d0ab1.jpg: 0.9960 объектов: 46
brick2-17-_jpg.rf.f64be790718963275d995fa4b939e546.jpg: 0.9821 объектов: 85
_barhat-2-500x500-500x500_jpg.rf.0ac56ded39cc32563a75454ad8649138.jpg: 0.9989 объектов: 28
383_jpg.rf.bbc945440a6bb651e33008744c19bcd2.jpg: 0.8869 объектов: 25
4-1-_jpg.rf.38692d9e8160c842930d08b826f87182.jpg: 0.9980 объектов: 104
full_4404479-0-1-_jpg.rf.3fffd358b0471fd970c05ddeac0945e5.jpg: 0.9956 объектов: 42
stack-bricks-construction-territory-near-future-house-72727138_jpg.rf.4dcad82f85499f905a9cf3aa286f0c44.jpg: 0.9869 объектов: 61
unnamed_jpg.rf.2843209a631061453c3a09b086990059.jpg: 0.9980 объектов: 35
stack-old-red-bricks-text